# Import Libraries

In [ ]:
from evalwrf.api import load_url_for_resource, save_data, save_json_from_URL
from evalwrf.preprocess.geosphere_interface import load_csv
from evalwrf.postprocess.postprocessing import plot_field, timeseries_station
from evalwrf.utils.windturbines import to_windturbine
import xarray as xr
import pandas as pd

# Save all available datasets

In [ ]:
save_json_from_URL("https://dataset.api.hub.geosphere.at/v1/datasets", "Datasets")

# Save Metadata for resource

In [ ]:
url = load_url_for_resource("../src/evalwrf/config/Datasets.json", resource="/grid/forecast/nwp-v1-1h-2500m")
save_json_from_URL(url / "metadata", filename="Metadata_nwp-v1-1h-2500m")

## Loading Multi-Dimensional Resources

In [ ]:
url

In [ ]:
save_data(
    url,
    filename="nwp.nc",
    params=dict(
        start="2026-04-02T00:00+00:00",
        end="2026-04-02T12:00+00:00",
        parameters=["t2m", "tcc", "rr_acc", "snow_acc"],
        bbox="45.0,8.0,50.0,18.0",
        # station_ids=15920,
        output_format="netcdf",
    ),
)

### Loading and plotting downloaded data

In [ ]:
da = xr.load_dataset("nwp.nc",decode_timedelta=True)

In [ ]:
plot_field(da["t2m"],cmap="jet")

## Downloading Timeseries Data

In [ ]:
url = load_url_for_resource("../src/evalwrf/config/Datasets.json", resource="klima-v2-10min")

In [ ]:
save_data(
    url,
    filename="murau_data.csv",
    params=dict(
        start="2026-03-10",
        end="2026-03-20",
        parameters=["TL", "RR"],
        station_ids=15920,
        output_format="csv",
    ),
)

### Plotting Station Timeseries data

In [ ]:
df = load_csv("murau_data.csv")

In [ ]:
fig = timeseries_station(df, y="tl", title="Timeseries Murau Temperature")

### Create wind turbine file(s) (.tbl)

In [ ]:
df = pd.read_excel("Windturbines.xlsx", sheet_name="Leistungskurven", usecols="A:H")

configs = {
    "V117": dict(sheet_name="Vestas", usecols="A:C"),
    "V122": dict(sheet_name="Vestas", usecols="D:F"),
    "V126": dict(sheet_name="Vestas", usecols="G:I"),
    "V162": dict(sheet_name="Vestas", usecols="J:L"),
    "V136": dict(sheet_name="Vestas", usecols="M:O"),
    "V150": dict(sheet_name="Vestas", usecols="P:R"),
    "V112": dict(sheet_name="Vestas", usecols="S:U"),
    "N149": dict(sheet_name="Nordex", usecols="A:C"),
    "N163": dict(sheet_name="Nordex", usecols="D:F"),
    "E82-E4": dict(sheet_name="Enercon", usecols="A:C"),
}

for k, v in configs.items():
    data: pd.DataFrame = pd.read_excel(
        "Windturbines.xlsx", skiprows=1, **v
    ).dropna()
    data.columns = [c.split(".")[0] for c in data.columns]
    attribute_data = df[df["Callname"] == k].drop_duplicates(subset="Callname")
    attribute_data = attribute_data.drop(columns=["Marke", "Name", "Callname"])

    for col in attribute_data:
        val = attribute_data[col].item()
        data.attrs[col] = val

    to_windturbine(
        df=data, filename=f"wind-turbine-{data.attrs['Index Value']}.tbl"
    )


### Loading WRF Data

In [ ]:
from pathlib import Path
from typing import Literal

def get_domain_files(foldername : str, domain : Literal["d01","d02","d03","d04","d05"],input : bool = False) -> list:
    prefix = "wrfout" if not input else "wrfin"
    return sorted(Path(foldername).glob(f"{prefix}*{domain}*"))
    
files = get_domain_files("...","d03")

In [ ]:
ds = xr.open_dataset(files[0])
ds